In [1]:
%run ./003_1_dq_rule_engine

StatementMeta(, a084b6d8-ee67-4630-ba20-077e80a42a05, 15, Finished, Available, Finished, True)

In [2]:
%run ./03_1_1_dq_rule_config

StatementMeta(, a084b6d8-ee67-4630-ba20-077e80a42a05, 16, Finished, Available, Finished, True)

In [3]:
from pyspark.sql import functions as F
from datetime import datetime

DB = "yelp_lakehouse.dbo"
LAYER = "silver"
DEBUG = True

dq_rule_tbl = f"{DB}.dq_rule_result"
dq_gate_tbl = f"{DB}.dq_table_gate"

pipeline_run_id = "manual_run"
run_ts = datetime.now()

StatementMeta(, a084b6d8-ee67-4630-ba20-077e80a42a05, 17, Finished, Available, Finished, False)

In [4]:
all_rule_results = []
all_gate_results = []

for table_name, rules in DQ_CONFIG.items():
    df = spark.table(f"{DB}.{table_name}")
    dq_run_id = new_dq_run_id()

    results = []

    for rule in rules:
        r = eval_standard_rule(
            df = df,
            rule = rule,
            dq_run_id = dq_run_id,
            pipeline_run_id = pipeline_run_id,
            layer = LAYER,
            table_name = table_name,
            run_ts = run_ts
        )
        results.append(r)
    
    gate = build_gate_result(
        results,
        dq_run_id,
        pipeline_run_id,
        LAYER,
        table_name,
        run_ts
    )

    all_rule_results.extend(results)
    all_gate_results.append(gate)

StatementMeta(, a084b6d8-ee67-4630-ba20-077e80a42a05, 18, Finished, Available, Finished, False)

In [5]:
rule_df = spark.createDataFrame(all_rule_results)
gate_df = spark.createDataFrame(all_gate_results)
(
    rule_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(dq_rule_tbl)
)

(
    gate_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(dq_gate_tbl)
)

StatementMeta(, a084b6d8-ee67-4630-ba20-077e80a42a05, 19, Finished, Available, Finished, False)

In [6]:
if DEBUG:
    display(rule_df)
    display(gate_df)

StatementMeta(, a084b6d8-ee67-4630-ba20-077e80a42a05, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, feb7225e-7dfc-41cf-86fe-01c1b61a0b0d)

SynapseWidget(Synapse.DataFrame, d6d272d0-5920-498a-bc3c-415ca5a0acc8)